# 02 — Composition by subtype and lineage

GC content and the four GC dinucleotide O/E ratios, compared between RSV-A and RSV-B and across lineages.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 120)
from rsv import io, composition


In [ ]:
d = pd.read_csv("../data/processed/rsv_features.tsv", sep="\t")
FEATS = ["gc_content","oe_CG","oe_GC","oe_GG","oe_CC"]
d.groupby("Subtype")[FEATS].agg(["mean","std"]).round(4)

### A vs B: is the difference significant? (Mann–Whitney U)

In [ ]:
from scipy.stats import mannwhitneyu
for f in FEATS:
    a=d[d.Subtype=="A"][f]; b=d[d.Subtype=="B"][f]
    u,p = mannwhitneyu(a,b)
    print(f"{f:10s} A={a.mean():.4f} B={b.mean():.4f}  p={p:.2e}")

### Composition across the most common lineages

In [ ]:
top = (d.groupby(["Subtype","Lineage"]).size().rename("n").reset_index()
        .sort_values("n",ascending=False).groupby("Subtype").head(8))
lin = d.merge(top[["Subtype","Lineage"]], on=["Subtype","Lineage"])
summ = lin.groupby(["Subtype","Lineage"])[FEATS+["gc_content"]].mean()
summ = lin.groupby(["Subtype","Lineage"]).agg(n=("gc_content","size"),
        gc=("gc_content","mean"), oe_CG=("oe_CG","mean")).round(4)
summ.to_csv("../results/composition_by_lineage_summary.tsv", sep="\t")
summ

In [ ]:
g = sns.catplot(data=lin, x="oe_CG", y="Lineage", col="Subtype", kind="box",
    sharey=False, height=4, aspect=1, color="#6baed6")
g.set_titles("RSV-{col_name}"); plt.show()